# Ops Decision Engine
# 04_rag_data_preparation.ipynb

In [12]:
import pandas as pd
import json
from pathlib import Path

In [13]:
RAW_PATH = Path("../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv")
OUTPUT_PATH = Path("../data/processed/rag_knowledge_base.jsonl")

In [14]:
df = pd.read_csv(RAW_PATH)

print("Raw dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Raw dataset shape: (28587, 16)

Columns:
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [15]:
df.head(5)

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [16]:
print("Language distribution:\n")
print(df["language"].value_counts(dropna=False))

Language distribution:

language
en    16338
de    12249
Name: count, dtype: int64


In [17]:
# Keep English only for now
df_en = df[df["language"] == "en"].copy()

print("English-only shape:", df_en.shape)

English-only shape: (16338, 16)


In [18]:
# Fill important fields
df_en["subject"] = df_en["subject"].fillna("").astype(str).str.strip()
df_en["body"] = df_en["body"].fillna("").astype(str).str.strip()
df_en["answer"] = df_en["answer"].fillna("").astype(str).str.strip()
df_en["type"] = df_en["type"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
df_en["queue"] = df_en["queue"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
df_en["priority"] = df_en["priority"].fillna("UNKNOWN").astype(str).str.strip().str.upper()

In [19]:
# Create issue description from subject + body
df_en["issue_description"] = (
    df_en["subject"] + " " + df_en["body"]
).str.strip()

print(df_en[["issue_description", "answer", "type", "queue", "priority"]].head(3))

                                   issue_description  \
1  Account Disruption Dear Customer Support Team,...   
2  Query About Smart Home System Integration Feat...   
3  Inquiry Regarding Invoice Details Dear Custome...   

                                              answer      type  \
1  Thank you for reaching out, <name>. We are awa...  INCIDENT   
2  Thank you for your inquiry. Our products suppo...   REQUEST   
3  We appreciate you reaching out with your billi...   REQUEST   

                   queue priority  
1      TECHNICAL SUPPORT     HIGH  
2  RETURNS AND EXCHANGES   MEDIUM  
3   BILLING AND PAYMENTS      LOW  


In [22]:
# Keep only rows that have useful issue text and useful resolution text
rag_df = df_en.copy()

rag_df = rag_df[
    (rag_df["issue_description"].str.len() > 20) &
    (rag_df["answer"].str.len() > 20)
].copy()

print("RAG candidate shape:", rag_df.shape)

RAG candidate shape: (16307, 17)


In [23]:
# Collect tags into one string
tag_cols = [col for col in rag_df.columns if col.startswith("tag_")]

def combine_tags(row):
    tags = []
    for col in tag_cols:
        value = row[col]
        if pd.notna(value) and str(value).strip() != "":
            tags.append(str(value).strip().lower())
    return tags

rag_df["tags"] = rag_df.apply(combine_tags, axis=1)

print(rag_df[["tags"]].head(3))

                                                tags
1    [account, disruption, outage, it, tech support]
2                   [product, feature, tech support]
3  [billing, payment, account, documentation, fee...


In [24]:
# Build retrieval text
# This is the text that will be embedded into the vector DB

def build_retrieval_text(row):
    tags_text = ", ".join(row["tags"]) if row["tags"] else "none"

    return (
        f"Issue: {row['issue_description']}\n"
        f"Type: {row['type']}\n"
        f"Queue: {row['queue']}\n"
        f"Priority: {row['priority']}\n"
        f"Resolution: {row['answer']}\n"
        f"Tags: {tags_text}"
    )

rag_df["retrieval_text"] = rag_df.apply(build_retrieval_text, axis=1)

print(rag_df["retrieval_text"].iloc[0])

Issue: Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?
Type: INCIDENT
Queue: TECHNICAL SUPPORT
Priority: HIGH
Resolution: Thank you for reaching out, <name>. We are aware of the outage affecting the centralized account management system, and our technical team is actively working to resolve the issue. In the meantime, we suggest using alternative methods to manage your account, with a focus on restoring service as quickly as possible. We will provide an update as soon as the serv

In [25]:
# Create final RAG schema
rag_kb = rag_df[
    [
        "issue_description",
        "answer",
        "type",
        "queue",
        "priority",
        "tags",
        "retrieval_text",
    ]
].copy()

rag_kb = rag_kb.reset_index(drop=True)
rag_kb["doc_id"] = rag_kb.index.map(lambda i: f"incident_{i+1}")
rag_kb["source"] = "incident_memory"

# Reorder columns
rag_kb = rag_kb[
    [
        "doc_id",
        "source",
        "retrieval_text",
        "issue_description",
        "answer",
        "type",
        "queue",
        "priority",
        "tags",
    ]
]

print("Final RAG KB shape:", rag_kb.shape)
rag_kb.head(3)

Final RAG KB shape: (16307, 9)


,doc_id,source,retrieval_text,issue_description,answer,type,queue,priority,tags
0,incident_1,incident_memory,Issue: Account Disruption Dear Customer Suppor...,"Account Disruption Dear Customer Support Team,...","Thank you for reaching out, <name>. We are awa...",INCIDENT,TECHNICAL SUPPORT,HIGH,"[account, disruption, outage, it, tech support]"
1,incident_2,incident_memory,Issue: Query About Smart Home System Integrati...,Query About Smart Home System Integration Feat...,Thank you for your inquiry. Our products suppo...,REQUEST,RETURNS AND EXCHANGES,MEDIUM,"[product, feature, tech support]"
2,incident_3,incident_memory,Issue: Inquiry Regarding Invoice Details Dear ...,Inquiry Regarding Invoice Details Dear Custome...,We appreciate you reaching out with your billi...,REQUEST,BILLING AND PAYMENTS,LOW,"[billing, payment, account, documentation, fee..."


In [26]:
# Rename answer -> resolution for cleaner semantics
rag_kb = rag_kb.rename(columns={"answer": "resolution"})

rag_kb.head(3)

,doc_id,source,retrieval_text,issue_description,resolution,type,queue,priority,tags
0,incident_1,incident_memory,Issue: Account Disruption Dear Customer Suppor...,"Account Disruption Dear Customer Support Team,...","Thank you for reaching out, <name>. We are awa...",INCIDENT,TECHNICAL SUPPORT,HIGH,"[account, disruption, outage, it, tech support]"
1,incident_2,incident_memory,Issue: Query About Smart Home System Integrati...,Query About Smart Home System Integration Feat...,Thank you for your inquiry. Our products suppo...,REQUEST,RETURNS AND EXCHANGES,MEDIUM,"[product, feature, tech support]"
2,incident_3,incident_memory,Issue: Inquiry Regarding Invoice Details Dear ...,Inquiry Regarding Invoice Details Dear Custome...,We appreciate you reaching out with your billi...,REQUEST,BILLING AND PAYMENTS,LOW,"[billing, payment, account, documentation, fee..."


In [27]:
# Sanity checks
print("Rows:", len(rag_kb))
print("Unique doc_id:", rag_kb["doc_id"].nunique())
print("Unique issue descriptions:", rag_kb["issue_description"].nunique())

print("\nPriority distribution:")
print(rag_kb["priority"].value_counts())

print("\nType distribution:")
print(rag_kb["type"].value_counts())

print("\nQueue distribution:")
print(rag_kb["queue"].value_counts().head(10))

Rows: 16307
Unique doc_id: 16307
Unique issue descriptions: 16307

Priority distribution:
priority
MEDIUM    6604
HIGH      6337
LOW       3366
Name: count, dtype: int64

Type distribution:
type
INCIDENT    6563
REQUEST     4653
PROBLEM     3389
CHANGE      1702
Name: count, dtype: int64

Queue distribution:
queue
TECHNICAL SUPPORT                  4731
PRODUCT SUPPORT                    3057
CUSTOMER SERVICE                   2408
IT SUPPORT                         1941
BILLING AND PAYMENTS               1590
RETURNS AND EXCHANGES               820
SERVICE OUTAGES AND MAINTENANCE     664
SALES AND PRE-SALES                 512
HUMAN RESOURCES                     348
GENERAL INQUIRY                     236
Name: count, dtype: int64


In [28]:
# Save as JSONL
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for _, row in rag_kb.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"Saved RAG knowledge base to: {OUTPUT_PATH}")

Saved RAG knowledge base to: ..\data\processed\rag_knowledge_base.jsonl


In [29]:
# Verify saved file by loading first few lines
sample_records = []
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        sample_records.append(json.loads(line))

sample_records

[{'doc_id': 'incident_1',
  'source': 'incident_memory',
  'retrieval_text': 'Issue: Account Disruption Dear Customer Support Team,\\n\\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\\n\\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?\nType: INCIDENT\nQueue: TECHNICAL SUPPORT\nPriority: HIGH\nResolution: Thank you for reaching out, <name>. We are aware of the outage affecting the centralized account management system, and our technical team is actively working to resolve the issue. In the meantime, we suggest using alternative methods to manage your account, with a focus on r